# Explore post-processed MovieLens data

Sanity-checks on the output of `preprocess_movielens.ipynb`: split sizes, id consistency, interaction distributions, and a peek at the simulator jsonl.

# 0. Import

In [1]:
import os
import json

import pandas as pd
from local_package.config.data import MOVIELENS_PROCESSED_DIR

# 1. Configuration

In [2]:
# --- config: point this at the same OUTPUT_DIR used in preprocess_movielens.ipynb ---
OUTPUT_DIR = MOVIELENS_PROCESSED_DIR / "chatbot"

# 2. Load outputs

In [3]:
df_train = pd.read_csv(OUTPUT_DIR / "train.tsv")
df_valid = pd.read_csv(OUTPUT_DIR / "valid.tsv")
df_test = pd.read_csv(OUTPUT_DIR / "test.tsv")
user_history = pd.read_csv(OUTPUT_DIR / "user_history.tsv")
products = pd.read_feather(OUTPUT_DIR / "products.ftr")

In [4]:
with open(OUTPUT_DIR / "map.json") as f:
    id_maps = json.load(f)

In [5]:
simulator_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("simulator_test_data_")]
simulator_path = OUTPUT_DIR / simulator_files[0]
simulator_data = [json.loads(line) for line in open(simulator_path)]

In [6]:
print("train:", df_train.shape)
print("valid:", df_valid.shape)
print("test:", df_test.shape)
print("user_history (train+valid):", user_history.shape)
print("products:", products.shape)
print("users in map:", len(id_maps["user"]))
print("items in map:", len(id_maps["item"]))
print("simulator records:", len(simulator_data))

train: (8100564, 2)
valid: (69814, 2)
test: (69814, 3)
user_history (train+valid): (8170378, 3)
products: (9888, 7)
users in map: 69814
items in map: 9888
simulator records: 900


## Sanity checks

In [7]:
# leave-one-out: every user should appear exactly once in valid and once in test
valid_counts = df_valid["UserID"].value_counts()
test_counts = df_test["UserID"].value_counts()
print("users with != 1 valid row:", (valid_counts != 1).sum())
print("users with != 1 test row:", (test_counts != 1).sum())

users with != 1 valid row: 0
users with != 1 test row: 0


In [8]:
# item ids in the splits should all exist in the product table
known_ids = set(products["id"])
for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    missing = (~df["MovieID"].isin(known_ids)).sum()
    print(f"{name}: {missing} item_ids missing from products table")

train: 0 item_ids missing from products table
valid: 0 item_ids missing from products table
test: 0 item_ids missing from products table


In [9]:
# id range should match the map sizes
print("max UserID in train:", df_train["UserID"].max(), "vs users in map:", len(id_maps["user"]))
print("max MovieID in train:", df_train["MovieID"].max(), "vs items in map:", len(id_maps["item"]))

max UserID in train: 69814 vs users in map: 69814
max MovieID in train: 9888 vs items in map: 9888


## Interaction distributions

In [10]:
all_inter = pd.concat([df_train, df_valid, df_test])
print("interactions per user:\n", all_inter.groupby("UserID").size().describe())
print("\ninteractions per item:\n", all_inter.groupby("MovieID").size().describe())

interactions per user:
 count    69814.000000
mean       118.030653
std        168.547473
min          5.000000
25%         31.000000
50%         60.000000
75%        133.000000
max       6169.000000
dtype: float64

interactions per item:
 count     9888.000000
mean       833.352751
std       2279.148708
min          5.000000
25%         30.000000
50%        116.000000
75%        532.000000
max      32379.000000
dtype: float64


## Product table

In [11]:
print("visited_num stats:")
print(products["visited_num"].describe())
print("\nmost-visited movies:")
print(products.sort_values("visited_num", ascending=False)[["Title", "visited_num"]].head(10))

visited_num stats:
count     9888.000000
mean       826.292273
std       2262.400042
min          3.000000
25%         30.000000
50%        114.500000
75%        527.000000
max      32194.000000
Name: visited_num, dtype: float64

most-visited movies:
                                                 Title  visited_num
292                                       Pulp Fiction        32194
584                          Silence of the Lambs, The        32113
351                                       Forrest Gump        31630
314                          Shawshank Redemption, The        30379
473                                      Jurassic Park        28828
450                                      Fugitive, The        27901
107                                         Braveheart        27017
256  Star Wars: Episode IV - A New Hope (a.k.a. Sta...        26924
580                         Terminator 2: Judgment Day        26708
147                                          Apollo 13        25436


In [12]:
print("category (primary genre) distribution:")
print(products["category"].value_counts().head(20))
print(f"\nmissing/placeholder description (no genres listed): {(products['description'] == 'No description').mean():.1%}")
print(f"missing release_date (no year in title): {products['release_date'].isna().mean():.1%}")

category (primary genre) distribution:
category
Drama          2883
Comedy         2833
Action         1403
Crime           543
Adventure       539
Horror          499
Documentary     365
Children        180
Animation       153
Thriller        113
Western          81
Sci-Fi           64
Romance          57
Mystery          44
Fantasy          42
Musical          42
Film-Noir        26
War              19
IMAX              1
Unknown           1
Name: count, dtype: int64

missing/placeholder description (no genres listed): 0.0%
missing release_date (no year in title): 0.0%


## Simulator jsonl peek

In [13]:
for rec in simulator_data[:5]:
    print("history:", rec["history"][:200])
    print("target: ", rec["target"])
    print()

history: Matrix, The; Celluloid Closet, The; Thin Blue Line, The; Stop Making Sense; When We Were Kings; Roger & Me; Everest; Thirty-Two Short Films About Glenn Gould; Microcosmos (Microcosmos: Le peuple de l'
target:  Buena Vista Social Club

history: Omen, The; Poseidon Adventure, The; Great Gatsby, The; Ryan's Daughter; Airport; Love Story; Towering Inferno, The; Rocky II; Benji; Last Tango in Paris (Ultimo tango a Parigi)
target:  Wuthering Heights

history: Strictly Ballroom; Mrs. Doubtfire; Milk Money; Singin' in the Rain; Wizard of Oz, The; Philadelphia Story, The; Sneakers; Thin Man, The; When Harry Met Sally...; It Happened One Night
target:  Pollyanna

history: Dirty Dozen, The; Longest Day, The; Astronaut's Wife, The; Like Water for Chocolate (Como agua para chocolate; Much Ado About Nothing; Crying Game, The; Jungle Fever; Dances with Wolves; Clear and Pre
target:  Mask of Zorro, The

history: Sabrina; Roman Holiday; Witness; Man in the Iron Mask, The; American President, T